# Bluebirdlite Sales vs. Price Time Series Analysis

This notebook follows the outlined workflow: visualize the log-sales and price series, study their autocorrelation structures, examine naive cross-correlations, perform prewhitening to avoid spurious relationships, and finally fit as well as diagnose a regression model linking sales to prices.

In [ ]:
# Imports and plotting configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima_process import arma_filter
import statsmodels.api as sm

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 14

data_path = Path("birdbluelite.dat.txt")
assert data_path.exists(), f"Data file not found at {data_path}"

In [ ]:
# Load and inspect the weekly log-sales and price series
data = pd.read_csv(data_path, sep=r"\s+")
data['week'] = pd.RangeIndex(start=1, stop=len(data) + 1, step=1, name='week')
data = data.set_index('week')

display(data.head())
display(data.describe().T)

In [ ]:
# Plot the time series to assess level, trend, and volatility structures
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(14, 8))
axes[0].plot(data.index, data['log_sales'], color='#1f77b4', linewidth=2)
axes[0].set_title('Weekly Log Sales')
axes[0].set_ylabel('log(sales)')
axes[1].plot(data.index, data['price'], color='#d62728', linewidth=2)
axes[1].set_title('Weekly Price')
axes[1].set_ylabel('Price')
axes[1].set_xlabel('Week')
plt.tight_layout()
fig.savefig('time_series_overview.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ACF and PACF for the original series
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_acf(data['log_sales'], lags=24, zero=False, ax=axes[0, 0])
axes[0, 0].set_title('ACF: Log Sales')
plot_pacf(data['log_sales'], lags=24, zero=False, method='ywm', ax=axes[0, 1])
axes[0, 1].set_title('PACF: Log Sales')
plot_acf(data['price'], lags=24, zero=False, ax=axes[1, 0])
axes[1, 0].set_title('ACF: Price')
plot_pacf(data['price'], lags=24, zero=False, method='ywm', ax=axes[1, 1])
axes[1, 1].set_title('PACF: Price')
plt.tight_layout()
fig.savefig('acf_pacf_levels.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# First-difference both series to reduce low-frequency components

diff_data = data.diff().dropna()
display(diff_data.head())
display(diff_data.describe().T)

In [ ]:
# ACF and PACF for the differenced series
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_acf(diff_data['log_sales'], lags=24, zero=False, ax=axes[0, 0])
axes[0, 0].set_title('ACF: Δ Log Sales')
plot_pacf(diff_data['log_sales'], lags=24, zero=False, method='ywm', ax=axes[0, 1])
axes[0, 1].set_title('PACF: Δ Log Sales')
plot_acf(diff_data['price'], lags=24, zero=False, ax=axes[1, 0])
axes[1, 0].set_title('ACF: Δ Price')
plot_pacf(diff_data['price'], lags=24, zero=False, method='ywm', ax=axes[1, 1])
axes[1, 1].set_title('PACF: Δ Price')
plt.tight_layout()
fig.savefig('acf_pacf_differences.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Naive cross-correlation between levels (subject to spurious peaks)

def lagged_corr(x, y, lag):
    if lag > 0:
        return x.iloc[lag:].corr(y.iloc[:-lag])
    if lag < 0:
        return x.iloc[:lag].corr(y.iloc[-lag:])
    return x.corr(y)

lag_max = 20
lags = np.arange(-lag_max, lag_max + 1)
ccf_values = [lagged_corr(data['price'], data['log_sales'], lag) for lag in lags]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(lags, ccf_values, color='#2ca02c')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Lag (Price leads → positive)')
ax.set_ylabel('Correlation')
ax.set_title('Naive Cross-Correlation between Price and Log Sales')
ax.set_xticks(range(-lag_max, lag_max + 1, 4))
plt.tight_layout()
fig.savefig('naive_ccf_levels.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Prewhitening: fit ARMA to ΔPrice, filter ΔLogSales with the same model, then inspect CCF
best_aic = np.inf
best_order = None
best_result = None
price_diff = diff_data['price']

for p in range(4):
    for q in range(4):
        if p == 0 and q == 0:
            continue
        try:
            model = ARIMA(price_diff, order=(p, 0, q), trend='n')
            result = model.fit()
            if result.aic < best_aic:
                best_aic = result.aic
                best_order = (p, q)
                best_result = result
        except Exception:
            continue

print(f"Selected ARMA({best_order[0]}, {best_order[1]}) for ΔPrice with AIC={best_aic:.2f}")

ar_params = getattr(best_result, 'arparams', np.array([]))
ma_params = getattr(best_result, 'maparams', np.array([]))
ar_poly = np.r_[1, -ar_params] if ar_params.size else np.array([1.0])
ma_poly = np.r_[1, ma_params] if ma_params.size else np.array([1.0])

filtered_price = arma_filter(price_diff.values, ar_poly, ma_poly)
filtered_sales = arma_filter(diff_data['log_sales'].values, ar_poly, ma_poly)

burn = max(len(ar_poly), len(ma_poly)) - 1
filtered_price = pd.Series(filtered_price[burn:], index=price_diff.index[burn:])
filtered_sales = pd.Series(filtered_sales[burn:], index=diff_data.index[burn:])

lags = np.arange(-lag_max, lag_max + 1)
prewhitened_ccf = [lagged_corr(filtered_price, filtered_sales, lag) for lag in lags]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(lags, prewhitened_ccf, color='#ff7f0e')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Lag (Filtered price leads → positive)')
ax.set_ylabel('Correlation')
ax.set_title('Prewhitened CCF between ΔPrice and ΔLogSales')
ax.set_xticks(range(-lag_max, lag_max + 1, 4))
plt.tight_layout()
fig.savefig('prewhitened_ccf.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Static regression: log_sales_t = beta0 + beta1 * price_t + error_t
X = sm.add_constant(data['price'])
ols_model = sm.OLS(data['log_sales'], X).fit()
print(ols_model.summary())
print(f"Durbin-Watson statistic: {sm.stats.durbin_watson(ols_model.resid):.2f}")

In [ ]:
# Diagnose regression residuals
residuals = ols_model.resid

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
axes[0].plot(residuals, color='#9467bd')
axes[0].set_title('OLS Residuals over Time')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Residual')
plot_acf(residuals, lags=24, zero=False, ax=axes[1])
axes[1].set_title('ACF: OLS Residuals')
plot_pacf(residuals, lags=24, zero=False, method='ywm', ax=axes[2])
axes[2].set_title('PACF: OLS Residuals')
plt.tight_layout()
fig.savefig('ols_residual_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()